# Shadow Agent Pro — Exploratory Data Analysis

Analyzes the labeled URL dataset used to train the detection models: class
balance, lexical feature distributions by class, and — after training —
the Random Forest's feature importances and the confusion matrix on the
held-out test set.

**Run this after generating a dataset** (`generate_sample_dataset.py` or
`download_real_dataset.py`) **and training a model** (`train.py`), since
the later cells load the trained artifacts. Run from the `ml-training/`
directory, or adjust the paths in the first code cell.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Jupyter sets the kernel's working directory to this notebook's own folder
# (ml-training/notebooks/), not wherever you launched `jupyter` from — so
# paths are resolved relative to THIS file's location, not the invocation
# directory. Two parents up from notebooks/ reaches the project root.
NOTEBOOK_DIR = Path.cwd()
ML_TRAINING_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = ML_TRAINING_DIR.parent

sys.path.append(str(PROJECT_ROOT / "backend"))
from ml.feature_extraction import extract_all_features, FEATURE_ORDER

DATA_PATH = ML_TRAINING_DIR / "datasets" / "urls_labeled_real.csv"   # or urls_labeled.csv for the synthetic set
ARTIFACTS_DIR = PROJECT_ROOT / "backend" / "ml" / "artifacts"

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 1. Load the dataset and check class balance

Real phishing/malware datasets are almost never perfectly balanced —
worth confirming what we're actually working with before trusting any
accuracy number.

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8", encoding_errors="replace")
print(f"{len(df):,} rows")
df["label"].value_counts().rename({0: "benign", 1: "malicious"})

In [ ]:
df["label"].value_counts().rename({0: "benign", 1: "malicious"}).plot(
    kind="bar", color=["#43a047", "#e53935"], title="Class balance"
)
plt.ylabel("count")
plt.show()

## 2. Extract features for a sample

Full feature extraction on the whole dataset can be slow (WHOIS/SSL are
live network calls) — `skip_whois=True` keeps this fast for exploratory
purposes. For a real accuracy number, use `train.py` directly rather than
this notebook.

In [ ]:
SAMPLE_SIZE = min(1000, len(df))
sample = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

feature_rows = [extract_all_features(u, skip_whois=True) for u in sample["url"]]
features_df = pd.DataFrame(feature_rows)[FEATURE_ORDER]
features_df["label"] = sample["label"].values
features_df.head()

## 3. Lexical feature distributions by class

Do malicious URLs actually look different from benign ones on the
features we engineered? This is the sanity check every feature in
`feature_extraction.py` should pass — if a feature shows no separation
here, it's probably not pulling its weight in the model.

In [ ]:
numeric_features = ["url_length", "num_dots", "num_hyphens", "num_digits", "entropy", "num_subdomains"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat in zip(axes.flat, numeric_features):
    for label, color, name in [(0, "#43a047", "benign"), (1, "#e53935", "malicious")]:
        subset = features_df[features_df["label"] == label][feat]
        ax.hist(subset, bins=20, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Binary feature prevalence by class

For flag-like features (IP-literal, shortener, HTTPS, etc.), a simple
bar comparison is more readable than a histogram.

In [ ]:
binary_features = ["has_ip_address", "is_shortener", "uses_https", "has_valid_ssl", "brand_in_subdomain_not_domain"]

prevalence = features_df.groupby("label")[binary_features].mean().rename(index={0: "benign", 1: "malicious"})
prevalence.T.plot(kind="bar", color=["#43a047", "#e53935"])
plt.ylabel("proportion = True")
plt.title("Binary feature prevalence by class")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Trained model — feature importance

Requires a trained model in `../backend/ml/artifacts/` (run `train.py`
first). Shows which engineered features the Random Forest actually
relies on most — a useful cross-check against the distributions above.

In [ ]:
model_path = ARTIFACTS_DIR / "random_forest_model.joblib"

if model_path.exists():
    model = joblib.load(model_path)
    importances = pd.Series(model.feature_importances_, index=FEATURE_ORDER).sort_values(ascending=False)

    importances.head(15).plot(kind="barh", color="#5b7cfa")
    plt.gca().invert_yaxis()
    plt.title("Top 15 feature importances (Random Forest)")
    plt.tight_layout()
    plt.show()
else:
    print(f"No trained model found at {model_path} — run train.py first.")

## 6. Confusion matrix on a held-out sample

A quick visual sanity check — for a full, properly-split evaluation see
the output of `train.py` itself, which reports this on the actual test
split used during training.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from ml.feature_extraction import FEATURE_ORDER as _FO

scaler_path = ARTIFACTS_DIR / "feature_scaler.joblib"

if model_path.exists() and scaler_path.exists():
    scaler = joblib.load(scaler_path)
    X = scaler.transform(features_df[_FO])
    y_true = features_df["label"]
    y_pred = model.predict(X)

    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["benign", "malicious"]).plot(cmap="Blues")
    plt.title("Confusion matrix (this notebook's sample, not the official test split)")
    plt.show()
else:
    print("Model/scaler not found — run train.py first.")

## Notes

- This notebook uses a **random sample**, not the official train/test
  split from `train.py` — treat any metric shown here as a sanity check,
  not the reportable number. Use `train.py`'s own printed evaluation for
  anything you cite in a report.
- `skip_whois=True` means domain-age features are not meaningfully
  populated here — re-run feature extraction without that flag (and
  expect it to be much slower) if you want to inspect that signal
  specifically.